# 00 – Consolidated Run

Single notebook: config → optional ingestion → full pipeline (with query rewriter and risk rules) → optional evaluation. Run this instead of 01–08 when you want one place to run everything.

In [1]:
# config: project root, paths, and flags
import sys
from pathlib import Path

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

from rag.config import get_settings

settings = get_settings()
RUN_INGESTION = True  # set False after first run to use existing index
RUN_EVALUATION = True  # set True to run retrieval + safety + citation checks at the end

print("Input dir:", settings.input_data_dir)
print("Persist dir:", settings.persist_directory)
print("Run ingestion:", RUN_INGESTION)
print("Run evaluation:", RUN_EVALUATION)

Input dir: input_data
Persist dir: data/chroma
Run ingestion: True
Run evaluation: True


**Rebuild Chroma after adding `clause_type`**  
If the index was created before `clause_type` was added: set **`RUN_INGESTION = True`** in the config cell, then run the config and ingestion cells. The ingestion cell will remove `data/chroma/` and re-index so the new collection includes `clause_type`.

In [2]:
# optional: run ingestion (parse input_data, chunk, index to Chroma + BM25)
if RUN_INGESTION:
    import shutil
    chroma_dir = ROOT / settings.persist_directory
    if chroma_dir.exists():
        shutil.rmtree(chroma_dir)
        print(f"Removed {chroma_dir} for fresh index with clause_type.")
    chroma_dir.mkdir(parents=True, exist_ok=True)
    from rag.ingestion.pipeline import run_ingestion
    chunks = run_ingestion(
        input_dir=ROOT / settings.input_data_dir,
        persist_directory=ROOT / settings.persist_directory,
        index_to_store=True,
    )
    print(f"Indexed {len(chunks)} clause chunks.")
else:
    print("Skipping ingestion (RUN_INGESTION=False). Using existing index.")

Removed /Users/jashsinghania/data/all_codes/rag_bharat/data/chroma for fresh index with clause_type.


Parsing documents: 100%|██████████| 4/4 [00:00<00:00, 662.14it/s]

Documents in index (verify SLA ingested): ['Data Processing Agreement', 'Nda Acme Vendor', 'Service Level Agreement', 'Vendor Services Agreement']
Indexed 25 clause chunks.


In [3]:
# build store and orchestrator (includes query rewriter, risk rule engine, validation)
from rag.retrieval.store import IndexStore
from rag.orchestration import Orchestrator

store = IndexStore(persist_directory=ROOT / settings.persist_directory)
orchestrator = Orchestrator(index_store=store)

if not store.get_chunks():
    raise RuntimeError("No indexed data. Set RUN_INGESTION=True and run the ingestion cell first.")

In [4]:
# Exact stored document values in Chroma (run this to verify filter strings)
res = store.collection.get(include=["metadatas"])
docs = [m["document"] for m in res["metadatas"] if m.get("document")]
print(set(docs))

{'Service Level Agreement', 'Vendor Services Agreement', 'Data Processing Agreement', 'Nda Acme Vendor'}


In [5]:
# full pipeline: multi-turn, cross-document, guardrail, section-level
QUERIES = [
    "What is the notice period for terminating the NDA?",
    "For how long?",
    "Is there any unlimited liability?",
    "What is the uptime commitment in the SLA?",
    "Are there conflicting governing laws across the contracts?",
    "Should I sign this NDA?",
]

for q in QUERIES:
    print("Query:", q)
    resp = orchestrator.run(q)
    print("Answer:", resp.direct_answer)
    if resp.cited_clauses:
        print("Citations:", [{"document": c.get("document"), "section": c.get("section"), "title": c.get("title")} for c in resp.cited_clauses])
    if resp.risk_flags:
        print("Risks:", resp.risk_flags)
    if getattr(orchestrator, "last_latency_ms", None):
        print("Latency (ms):", orchestrator.last_latency_ms)
    print()

Query: What is the notice period for terminating the NDA?
Intent output: intent_type='termination' document_focus=['Nda Acme Vendor'] is_legal_advice=False requires_clarification=False refined_query='What is the notice period for terminating the NDA?' topic='termination'
intent.document_focus: ['Nda Acme Vendor']
Applying filter: {'document': 'Nda Acme Vendor'}
>>> CUSTOM RETRIEVE FUNCTION EXECUTED <<<
Retrieved docs: ['Nda Acme Vendor']
Answer: Either party may terminate this Agreement with thirty (30) days written notice.
(Source: Nda Acme Vendor, Section 3 - Term and Termination)
Citations: [{'document': 'Nda Acme Vendor', 'section': '3', 'title': 'Term and Termination'}]
Risks: [{'level': 'LOW', 'message': 'No high-risk patterns identified.'}]
Latency (ms): {'intent_ms': 0.05, 'rewrite_ms': 0.0, 'retrieval_ms': 57.42, 'clause_analysis_ms': 17082.29, 'risk_ms': 0.45, 'compose_ms': 0.82, 'total_ms': 17141.17}

Query: For how long?
Intent output: intent_type='clause_retrieval' documen

In [6]:
# verbose run: print each pipeline state and prompt names (set INCLUDE_PROMPTS=True to print prompt content)
INCLUDE_PROMPTS = False
VERBOSE_QUERIES = [
    "What is the notice period for terminating the NDA?",
    "For how long?",
]

for q in VERBOSE_QUERIES:
    print("=" * 60)
    print("Query:", q)
    state = orchestrator.run_verbose(q, include_prompt_content=INCLUDE_PROMPTS)
    print("Intent:", state["intent"])
    print("Rewritten query:", state["rewritten_query"])
    print("Retrieved (doc / section / title / snippet):")
    for r in state["retrieved_summary"] or []:
        print(" ", r)
    print("Clause analysis:", state["clause_analysis"])
    print("Risk:", state["risk"])
    print("Response:", state["response"].direct_answer if state["response"] else None)
    if state.get("latency_ms"):
        print("Latency (ms):", state["latency_ms"])
    if INCLUDE_PROMPTS and "prompts" in state:
        for name, content in state["prompts"].items():
            print(f"--- Prompt: {name} (first 300 chars) ---\n{content[:300]}...")
    print()

Query: What is the notice period for terminating the NDA?
Intent output: intent_type='termination' document_focus=['Nda Acme Vendor'] is_legal_advice=False requires_clarification=False refined_query='What is the notice period for terminating the NDA?' topic='termination'
intent.document_focus: ['Nda Acme Vendor']
Applying filter: {'document': 'Nda Acme Vendor'}
>>> CUSTOM RETRIEVE FUNCTION EXECUTED <<<
Retrieved docs: ['Nda Acme Vendor']
Intent: intent_type='termination' document_focus=['Nda Acme Vendor'] is_legal_advice=False requires_clarification=False refined_query='What is the notice period for terminating the NDA?' topic='termination'
Rewritten query: What is the notice period for terminating the NDA?
Retrieved (doc / section / title / snippet):
  {'document': 'Nda Acme Vendor', 'section': '3', 'title': 'Term and Termination', 'snippet': 'This Agreement shall remain in effect for three (3) years from the effective date.\nEither party may terminate this Agreement with thirty (30) 

In [7]:
# optional: evaluation (retrieval recall/precision/MRR/avg_rank, refusal, citation presence & correctness, risk, hallucination, multi-turn, cross-doc)
if RUN_EVALUATION:
    from rag.retrieval.hybrid_search import HybridRetrieval
    from rag.orchestration import normalize_document_focus, resolve_document_focus
    from rag.evaluation import (
        run_retrieval_eval,
        has_citation,
        citation_correct,
        refusal_for_legal_advice,
        answer_matches_gold,
        risk_matches,
        detect_hallucination,
        multi_turn_memory_ok,
        cross_document_mentions,
    )

    retrieval = HybridRetrieval(store=store, top_k=5)
    all_doc_names = store.get_all_documents()

    def retrieval_fn(query: str):
        intent = orchestrator.intent_agent.classify(query)

        raw_focus = intent.document_focus or []
        normalized_focus = normalize_document_focus(raw_focus) if raw_focus else []
        document_filter = resolve_document_focus(normalized_focus, all_doc_names) if normalized_focus else None

        hard_clause_type = intent.topic if intent.topic in (
            "termination",
            "liability",
            "confidentiality",
            "governing_law",
            "indemnification",
            "uptime",
        ) else None

        return retrieval.search(
            query,
            document_filter=document_filter,
            hard_clause_type=hard_clause_type,
            top_k=5,
        )

    RETRIEVAL_TEST_QUERIES = [
        {"query": "What is the notice period for terminating the NDA?", "relevant": [{"document": "Nda Acme Vendor", "section": "3"}]},
        {"query": "Data breach notification deadline", "relevant": [{"document": "Data Processing Agreement", "section": "3"}]},
        {"query": "What is the uptime commitment in the SLA?", "relevant": [{"document": "Service Level Agreement", "section": "Service Availability"}]},
    ]
    metrics = run_retrieval_eval(retrieval_fn, RETRIEVAL_TEST_QUERIES)
    print("Retrieval:", metrics)

    legal_resp = orchestrator.run("Should I sign this NDA?")
    print("Legal-advice refusal:", refusal_for_legal_advice(legal_resp))

    normal_resp, normal_ret = orchestrator.run_with_retrieval("What is the uptime commitment in the SLA?")
    print("Has citation:", has_citation(normal_resp))
    print("Citation correct (SLA expected):", citation_correct(normal_resp, "SLA", "Service Credits"))

    # Answer faithfulness (no embedding here; key phrase + numeric)
    gold_uptime = "Service credits as defined in Schedule B"
    match = answer_matches_gold(normal_resp.direct_answer or "", gold_uptime)
    print("Answer vs gold:", match)

    # Risk match (example: expect no high risk for uptime question)
    risk_sc = risk_matches(normal_resp.risk_flags or [], [], penalize_false_positive=True)
    print("Risk score (no expected risks):", risk_sc)

    # Hallucination check: answer vs retrieved text
    retrieved_text = " ".join(r.chunk.clause_text or "" for r in normal_ret)
    is_hall, suspicious = detect_hallucination(normal_resp.direct_answer or "", retrieved_text)
    print("Hallucination:", is_hall, "suspicious:", suspicious)

    # Multi-turn: first "Do confidentiality obligations survive termination?" then "For how long?"
    r1 = orchestrator.run("Do confidentiality obligations survive termination?")
    r2 = orchestrator.run("For how long?")
    mt = multi_turn_memory_ok("", r1, "", r2, required_phrases=["five (5) years", "five"], required_doc_in_citations="NDA")
    print("Multi-turn memory:", mt)

    # Cross-document: "Are there conflicting governing laws?"
    cross_resp = orchestrator.run("Are there conflicting governing laws across the contracts?")
    cross = cross_document_mentions(cross_resp, ["California", "England", "GDPR", "governing"])
    print("Cross-document mentions:", cross)
    if getattr(orchestrator, "last_latency_ms", None):
        print("Latency (ms) last run:", orchestrator.last_latency_ms)
else:
    print("Skipping evaluation (RUN_EVALUATION=False).")

Intent output: intent_type='termination' document_focus=['Nda Acme Vendor'] is_legal_advice=False requires_clarification=False refined_query='What is the notice period for terminating the NDA?' topic='termination'
intent.document_focus: ['Nda Acme Vendor']
>>> CUSTOM RETRIEVE FUNCTION EXECUTED <<<
Intent output: intent_type='data_breach' document_focus=['Data Processing Agreement'] is_legal_advice=False requires_clarification=False refined_query='Data breach notification deadline' topic='general'
intent.document_focus: ['Data Processing Agreement']
>>> CUSTOM RETRIEVE FUNCTION EXECUTED <<<
Intent output: intent_type='service_availability' document_focus=['Service Level Agreement'] is_legal_advice=False requires_clarification=False refined_query='What is the uptime commitment in the SLA?' topic='service_level'
intent.document_focus: ['Service Level Agreement']
>>> CUSTOM RETRIEVE FUNCTION EXECUTED <<<
Retrieval: {'recall_at_k': 0.6666666666666666, 'precision_at_k': 0.39999999999999997, 

In [8]:
# print("All documents in vector store:")
# print(store.get_all_documents())
# data = store.collection.get(include=["metadatas"])
# print("First 20 metadatas:")
# print(data["metadatas"][:20])

In [9]:
# print("Search for uptime without filter:")
# results = store.similarity_search("uptime", k=5)
# for r in results:
#     print(r.document, r.section, r.title)

In [10]:
# Step 1 — Confirm What’s Actually Stored

# Run this EXACT debug:

all_docs = store.collection.get(where={"document": "Service Level Agreement"})
print("Matched count:", len(all_docs["ids"]))


# Also try this:
all_docs_like = store.collection.get()
unique_docs = set(m["document"] for m in all_docs_like["metadatas"])
print("Unique document names in DB:", unique_docs)

Matched count: 6
Unique document names in DB: {'Service Level Agreement', 'Vendor Services Agreement', 'Data Processing Agreement', 'Nda Acme Vendor'}


In [11]:
print("Persist dir:", store.persist_directory)
print("Collection name:", store.collection.name)
print("Collection count:", store.collection.count())
docs = store.collection.get(where={"document": "Service Level Agreement"})
print("SLA direct get count:", len(docs["ids"]))

Persist dir: /Users/jashsinghania/data/all_codes/rag_bharat/data/chroma
Collection name: legal_clauses
Collection count: 25
SLA direct get count: 6
